In [1]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from neuroconv.datainterfaces import SbxImagingInterface

In [ ]:
import subprocess
import os
import getpass

# 1. Securely get your password once at the start
password = getpass.getpass("Enter remote SSH password: ")

# List of sessions to process
sessions = ["session_01", "session_02", "session_03"]
remote_user = "username"
remote_host = "192.168.1.100"
remote_path = "/path/to/imaging/data/"
local_tmp_path = "./tmp_data/"

for session in sessions:
    print(f"--- Starting {session} ---")
    
    # 2. Programmatically provide password using sshpass
    rsync_cmd = [
        "sshpass", "-p", password, 
        "rsync", "-avz", "--progress",
        f"{remote_user}@{remote_host}:{remote_path}{session}/",
        local_tmp_path
    ]
    
    try:
        # Download the session
        subprocess.run(rsync_cmd, check=True)
        
        # 3. YOUR CONVERSION CODE HERE
        # (e.g., Suite2p or NeuroConv commands for this session)
        print(f"Converting {session} to NWB...")
        
        # 4. Conserve disk space: Remove raw data after successful conversion
        # subprocess.run(["rm", "-rf", local_tmp_path], check=True)
        # os.makedirs(local_tmp_path, exist_ok=True)
        
    except subprocess.CalledProcessError as e:
        print(f"Error processing {session}: {e}")
        break 


In [3]:
OPHYS_DATA_PATH = Path('/mnt/BigDisk/2P_scratch/4467331.1/29_11_2020/YMaze_LNovel')

In [5]:
path_to_save_nwbfile = OPHYS_DATA_PATH / 'ymaze.nwb'

In [7]:


file_path = OPHYS_DATA_PATH / "YMaze_LNovel_001_003.sbx"
interface = SbxImagingInterface(file_path=file_path, verbose=False)

metadata = interface.get_metadata()
# For data provenance we add the time zone information to the conversion
session_start_time = datetime(2020, 1, 1, 12, 30, 0, tzinfo=ZoneInfo("US/Pacific"))
metadata["NWBFile"].update(session_start_time=session_start_time)
# Add subject information (required for DANDI upload)
metadata["Subject"] = dict(subject_id="subject1", 
                           species="Mus musculus", 
                           sex="M", 
                           age="P30D")

# Choose a path for saving the nwb file and run the conversion
nwbfile_path = f"{path_to_save_nwbfile}"
interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

NotImplementedError: The SbxImagingExtractor does not currently support multiple color channels or 3-dimensional depth.If you with to request either of these features, please do so by raising an issue at https://github.com/catalystneuro/roiextractors/issues

In [ ]:
import pandas as pd
from pynwb import NWBHDF5IO
from pynwb.behavior import BehavioralTimeSeries, TimeSeries

# 1. Setup - assume 'df' is your pandas dataframe from the sqlite file
# df = pd.read_sql_query("SELECT * FROM behavior_table", sqlite_conn)
timestamp_column = 'timestamps' # Update this to your actual column name

with NWBHDF5IO('existing_data.nwb', mode='a') as io:
    nwbfile = io.read()
    
    # 2. Get or create the behavior module
    if 'behavior' not in nwbfile.processing:
        beh_module = nwbfile.create_processing_module('behavior', 'behavioral data')
    else:
        beh_module = nwbfile.processing['behavior']

    # 3. Create a container to hold the multiple time series
    beh_ts_container = BehavioralTimeSeries(name='sqlite_behavior_data')

    # 4. Iterate through columns and add to container
    for col in df.columns:
        if col == timestamp_column:
            continue
            
        # Create TimeSeries for each behavioral metric (e.g., licks, speed)
        ts = TimeSeries(
            name=col,
            data=df[col].values,
            timestamps=df[timestamp_column].values,
            unit='arbitrary', # Change to specific units if known (e.g., 'm/s')
            description=f"Behavioral metric: {col}"
        )
        beh_ts_container.add_time_series(ts)

    # 5. Add container to the module and save
    beh_module.add(beh_ts_container)
    io.write(nwbfile)

In [8]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from neuroconv import ConverterPipe
from neuroconv.datainterfaces import TiffImagingInterface, Suite2pSegmentationInterface


folder_path= OPHYS_DATA_PATH / "YMaze_LNovel_001_003" / "suite2p"
interface_suite2p = Suite2pSegmentationInterface(folder_path=folder_path, verbose=False)

# Now that we have defined the two interfaces we pass them to the ConverterPipe which will coordinate the
# concurrent conversion of the data
metadata = interface_suite2p.get_metadata()

# For data provenance we add the time zone information to the conversion
session_start_time = datetime(2020, 1, 1, 12, 30, 0, tzinfo=ZoneInfo("US/Pacific"))
metadata["NWBFile"].update(session_start_time=session_start_time)
# Add subject information (required for DANDI upload)
metadata["Subject"] = dict(subject_id="subject1", species="Mus musculus", sex="M", age="P30D")

# Choose a path for saving the nwb file and run the conversion
nwbfile_path = f"{path_to_save_nwbfile}"
interface_suite2p.run_conversion(nwbfile_path=nwbfile_path,  metadata=metadata)

/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/roiextractors/extractors/suite2p/suite2psegmentationextractor.py:96: UserWarning: More than one channel is detected! Please specify which channel you wish to load with the `channel_name` argument. To see what channels are available, call `Suite2pSegmentationExtractor.get_available_channels(folder_path=...)`.
  warn(
/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/neuroconv/datainterfaces/ophys/suite2p/suite2pdatainterface.py:225: FutureWarning: The 'stub_frames' parameter is deprecated and will be removed on or after February 2026. Use 'stub_samples' instead.
  super().add_to_nwbfile(


In [9]:
metadata

DeepDict(
{'NWBFile': {'session_description': '',
  'identifier': '1e3eed78-9225-4fc0-a95c-4701f1f39a75',
  'source_script': 'Created using NeuroConv v0.9.1',
  'source_script_file_name': '/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/neuroconv/basedatainterface.py',
  'session_start_time': datetime.datetime(2020, 1, 1, 12, 30, tzinfo=zoneinfo.ZoneInfo(key='US/Pacific'))},
 'Ophys': {'Device': [{'name': 'Microscope'}],
  'ImagingPlane': [{'name': 'ImagingPlaneChan1Plane0',
    'description': 'The plane or volume being imaged by the microscope.',
    'excitation_lambda': nan,
    'indicator': 'unknown',
    'location': 'unknown',
    'device': 'Microscope',
    'optical_channel': [{'name': 'OpticalChannel',
      'emission_lambda': nan,
      'description': 'An optical channel of the microscope.'}]}],
  'Fluorescence': {'name': 'Fluorescence',
   'BackgroundPlaneSegmentation': {'neuropil': {'name': 'neuropil',
     'description': 'Array of neuropil traces.',
     'unit'

In [11]:
import suite2p
from suite2p.io import nwb

In [12]:
nwb.save_nwb(OPHYS_DATA_PATH / "YMaze_LNovel_001_003" / "suite2p")

root pynwb.file.NWBFile at 0x140386183063648
Fields:
  file_create_date: [datetime.datetime(2026, 2, 3, 21, 12, 51, 551700, tzinfo=tzlocal())]
  identifier: /media/mplitt/BigDisk/2P_scratch/4467331.1/29_11_2020/YMaze_LNovel/YMaze_LNovel_001_003
  session_description: suite2p_proc
  session_start_time: 2021-03-12 15:57:26.771677-08:00
  timestamps_reference_time: 2021-03-12 15:57:26.771677-08:00



/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/hdmf/container.py:542: UserWarning: The linked table for DynamicTableRegion 'rois' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()
/home/mplitt/mambaforge/envs/stx3/lib/python3.10/site-packages/suite2p/io/nwb.py:494: FutureWarning: Images.__init__: Using positional arguments for this method is discouraged and will be deprecated in a future major release. Please use keyword arguments to ensure future compatibility.
  images = Images("Backgrounds_%d" % iplane)


In [13]:
import pynwb